# BUS-BRA preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.bus_bra import BUSBRA, csv_save_path
from utils.preprocessing import UNKNOWN, birads_mapping

ds = BUSBRA()
ds.set_dataset_name("bus-bra")
ds.data_df.head()

### Step 1 — map raw BI-RADS codes to harmonized labels

In [ ]:
ds.data_df["birads"] = ds.data_df["birads"].apply(lambda x: birads_mapping.get(x, UNKNOWN))
ds.data_df["birads"].value_counts(dropna=False)

### Step 2 — drop ambiguous single-side laterality

In [ ]:
ds.data_df["side"] = ds.data_df["side"].replace("single", None)
ds.data_df["side"].value_counts(dropna=False)

## Build one exam record

In [ ]:
row = next(ds.data_df.itertuples(index=False))
exam = ds.process_example(row)
exam

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"rows before per-exam processing: {len(ds.data_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))